# RAG Evaluator

https://docs.langchain.com/langsmith/evaluate-rag-tutorial

Code: https://github.com/langchain-ai/langsmith-cookbook/tree/main/testing-examples/rag_eval

RAGAS - https://github.com/langchain-ai/langsmith-cookbook/blob/main/testing-examples/ragas/ragas.ipynb

In [ ]:
from dotenv import load_dotenv
import os

# Load environment variables FIRST - critical for LangSmith authentication
load_dotenv(dotenv_path='../.env')

# Set base URL and API key for OpenAI
base_url = ""
api_key = os.environ['UNIFIED_LLM_KEY']

os.environ["LANGSMITH_PROJECT"] = "Test"
os.environ["LANGSMITH_TRACING"] = "true"
# Verify LangSmith API key is loaded
if os.environ.get('LANGSMITH_API_KEY'):
    print("✅ LangSmith API key loaded successfully")
else:
    print("⚠️  WARNING: LANGSMITH_API_KEY not found!")
    print("   Please add LANGSMITH_API_KEY to: AGENTS/LANGCHAIN_GRAPH_SMITH/LANGSMITH/.env")

## Create DataSet

In [ ]:
from langsmith import Client


# QA
inputs = [
    "My LCEL map contains the key 'question'. What is the difference between using itemgetter('question'), lambda x: x['question'], and x.get('question')?",
    "How can I make the output of my LCEL chain a string?",
    "How can I run two LCEL chains in parallel and write their output to a map?",
]

outputs = [
    "Itemgetter can be used as shorthand to extract specific keys from the map. In the context of a map operation, the lambda function is applied to each element in the input map and the function returns the value associated with the key 'question'. (get) is safer for accessing values in a dictionary because it handles the case where the key might not exist.",
    "Use StrOutputParser. from langchain_openai import ChatOpenAI; from langchain_core.prompts import ChatPromptTemplate; from langchain_core.output_parsers import StrOutputParser; prompt = ChatPromptTemplate.from_template('Tell me a short joke about {topic}'); model = ChatOpenAI(model='gpt-3.5-turbo') #gpt-4 or other LLMs can be used here; output_parser = StrOutputParser(); chain = prompt | model | output_parser",
    # ... additional outputs
]

qa_pairs = [{"question": q, "answer": a} for q, a in zip(inputs, outputs)]

client = Client()
dataset_name = "RAGAS_QA_LCEL"

dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="QA pairs about LCEL."
)

client.create_examples(
    inputs=[{"question": q} for q in inputs],
    outputs=[{"answer": a} for a in outputs],
    dataset_id=dataset.id,
)

In [ ]:
todays_llm_runs = client.list_runs(
    project_name="Test",
    run_type="llm",
)

print(todays_llm_runs)

In [ ]:
datasets = client.list_datasets()
# Iterate over the datasets (optional)
for dataset in datasets:
    print(f"Dataset Name: {dataset.name}, ID: {dataset.id}")

In [ ]:
from langchain_openai import ChatOpenAI

In [ ]:
### INDEX

from bs4 import BeautifulSoup as Soup
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders.recursive_url_loader import RecursiveUrlLoader

url = "https://python.langchain.com/v0.1/docs/expression_language/"
loader = RecursiveUrlLoader(url=url, max_depth=20, extractor=lambda x: Soup(x, "html.parser").text)
docs = loader.load()
full_doc_text = ' ---- '.join([d.page_content for d in docs])

# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=4000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

In [ ]:
# Import necessary libraries
import sys
sys.path.append('/Users/vinotganesan/Learning/LLM & AGENTS/RAG')

from ml_server_embedding import get_embeddings
# Embed
vectorstore = Chroma.from_documents(documents=splits, embedding=get_embeddings())

# Index
retriever = vectorstore.as_retriever()

In [ ]:
### RAG

import openai
from langsmith import traceable
from langsmith.wrappers import wrap_openai
from langchain_core.prompts import PromptTemplate
from langchain_community.chat_models import ChatOllama
from langchain_core.output_parsers import StrOutputParser

class RagBot:
    """
    A class to interface with retrieval-augmented generation (RAG) models from different providers
    such as OpenAI or Ollama, utilizing a retriever for document-based context.
    """

    def __init__(
        self,
        retriever,
        provider: str = "openai",
        model: str = "gpt-4o",
        use_vectorstore: bool = True,
    ):
        """
        Initializes the RagBot with a retriever, provider information, model details, and configuration
        to use a vector store for document retrieval.

        Args:
        retriever: The document retriever instance.
        provider (str): The provider of the RAG model ('openai' or 'ollama').
        model (str): The model identifier used by the provider.
        use_vectorstore (bool): Flag to determine whether to use vectorstore for document retrieval.
        """
        self._retriever = retriever
        self._provider = provider
        self._model = model
        self._use_vectorstore = use_vectorstore
        if provider == "openai":
            self._client = wrap_openai(openai.Client(base_url=base_url, api_key=api_key))
        elif provider == "ollama":
            self._client = ChatOllama(model=model, temperature=0)

    @traceable()
    def retrieve_docs(self, question):
        """
        Retrieves documents based on the input question, using either a vectorstore or full context.

        Args:
        question (str): The question to retrieve documents for.

        Returns:
        list: A list of documents relevant to the question or the full context (as a string).
        """
        if self._use_vectorstore:
            return self._retriever.invoke(question)
        else:
            return full_doc_text

    @traceable()
    def get_answer(self, question: str):
        """
        Generates an answer for a given question by using RAG, leveraging both the retriever
        and the provider's model capabilities.

        Args:
        question (str): The user's question to answer.

        Returns:
        dict: A dictionary containing the 'answer' and 'contexts' (related documents).
        """
        similar = self.retrieve_docs(question)
        if self._provider == "openai":
            "OpenAI RAG"
            response = self._client.chat.completions.create(
                model=self._model,
                messages=[
                    {
                        "role": "system",
                        "content": "You are a helpful AI code assistant with expertise in LCEL.\n"
                        " Use the following docs to produce a concise code solution to the user question.\n"
                        " Use three sentences maximum and keep the answer concise. \n"
                        f"## Docs\n\n{similar}",
                    },
                    {"role": "user", "content": question},
                ],
            )
            response_str = response.choices[0].message.content

        elif self._provider == "ollama":
            "Ollama RAG"
            prompt = PromptTemplate(
                template="""You are a helpful AI code assistant with expertise in LCEL.
                Use the following docs to produce a concise code solution to the user question.
                If you don't know the answer, just say that you don't know.
                Use three sentences maximum and keep the answer concise.
                Question: {question}
                Context: {context}
                Answer: """,
                input_variables=["question", "context"],
            )
            rag_chain = prompt | self._client | StrOutputParser()
            response_str = rag_chain.invoke({"context": similar, "question": question})

        return {
            "answer": response_str,
            "contexts": [str(doc) for doc in similar],
        }

In [ ]:
def predict_rag_answer_oai(example: dict):
    """Use this for answer evaluation"""
    rag_bot = RagBot(retriever, provider="openai", model="gpt-4o")
    response = rag_bot.get_answer(example["question"])
    return {"answer": response["answer"]}


def predict_rag_answer_o4_mini(example: dict):
    """Use this for answer evaluation"""
    rag_bot = RagBot(retriever, provider="openai", model="o4-mini")
    response = rag_bot.get_answer(example["question"])
    return {"answer": response["answer"]}


def predict_rag_answer_claude(example: dict):
    """Use this for answer evaluation"""
    rag_bot = RagBot(retriever, provider="openai", model="claude-opus-4-6")
    response = rag_bot.get_answer(example["question"])
    return {"answer": response["answer"]}

In [ ]:
from langsmith.schemas import Run, Example
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def answer_evaluator(run:Run, example: Example):
    rag_answer = run.outputs.get("answer", "").strip()
    reference_answer = example.outputs.get("answer","").strip()
    question = example.inputs.get("question", "")

    class GradeAnswer(BaseModel):
        """A numerical score for answer accuracy."""
        score: int = Field(
            description="Answer matches the grond truth, score from 1 to 10"
        )

    # LLM with function call, use highest capacity model
    llm = ChatOpenAI(model="gpt-5", temperature=0, base_url=base_url, api_key=api_key)
    structured_llm_grader = llm.with_structured_output(GradeAnswer)

    # Prompt
    system = """Is the Assistant's Answer grounded in and similar to the Ground Truth answer. Note that we do not expect all of the text
            in code solution examples to be identical. We expect (1) code imports to be identical if the same import is used. (2) But, it is
            ok if there are differences in the implementation itself. The main point is that the same concept is employed. A score of 1 means
            that the Assistant answer is not at all conceptically grounded in and similar to the Ground Truth answer. A score of 5 means  that the Assistant
            answer contains some information that is conceptically grounded in and similar to the Ground Truth answer. A score of 10 means that the
            Assistant answer is fully conceptically grounded in and similar to the Ground Truth answer."""

    grade_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system),
            (
                "human",
                "Ground Truth answer: \n\n {reference} \n\n Assistant's Answer: {prediction}",
            ),
        ]
    )

    answer_grader = grade_prompt | structured_llm_grader
    score = answer_grader.invoke({"reference": reference_answer, "prediction": rag_answer})
    print('score : ', score)
    return {"key": "answer_accuracy", "score": int(score.score) / 10}

In [ ]:
from langsmith.evaluation import evaluate

evaluate_4o = evaluate(
    predict_rag_answer_oai,
    data=dataset_name,
    evaluators = [answer_evaluator],
    experiment_prefix="rag-qa-gpt-4o",
    metadata={"variant": "LCEL context, gpt-4"},
)

evaluate_4o_mini = evaluate(
    predict_rag_answer_o4_mini,
    data=dataset_name,
    evaluators = [answer_evaluator],
    experiment_prefix="rag-qa-gpt-4o-mini",
    metadata={"variant": "LCEL context, gpt-4o-mini"},
)

evaluate_claude = evaluate(
    predict_rag_answer_claude,
    data=dataset_name,
    evaluators = [answer_evaluator],
    experiment_prefix="rag-qa-claude-opus-4-6",
    metadata={"variant": "LCEL context, claude-opus-4-6"},
)

In [ ]:
from typing import Optional, Sequence

def multi_model_ranking_evaluator(runs: Sequence[Run], example: Optional[Example] = None) -> dict:
    """
    Rank models using existing 'answer_accuracy' scores from their evaluations.

    Extracts answer_accuracy feedback that was already computed.
    """
    from langsmith import Client

    client = Client()
    run_ids = [str(run.id) for run in runs]
    scores = []

    # Get existing answer_accuracy score for each run
    for run_id in run_ids:
        # Fetch feedback for this run
        feedbacks = list(client.list_feedback(run_ids=[run_id]))

        # Find answer_accuracy score
        answer_accuracy = None
        for feedback in feedbacks:
            if feedback.key == "answer_accuracy":  # or "answer_evaluator"
                answer_accuracy = feedback.score
                break

        # Use existing score or default to 0.0
        scores.append(answer_accuracy if answer_accuracy is not None else 0.0)

    return {
        "key": "multi_model_ranking",
        "scores": {run_id: score for run_id, score in zip(run_ids, scores)},
        "comment": f"Reused answer_accuracy scores: {[f'{s:.2f}' for s in scores]}"
    }

In [ ]:
from langsmith.evaluation import evaluate_comparative

comparison_results = evaluate_comparative(
    [evaluate_4o.experiment_name, evaluate_4o_mini.experiment_name, evaluate_claude.experiment_name],
    evaluators = [multi_model_ranking_evaluator],
    experiment_prefix="pairwise_comparison_gpt4o_vs_o4_mini_vs_claude_opus_4.6",
    metadata={
        "comparison": "GPT-4o vs o4-mini vs claude opus 4.6",
        "criteria": "RAG Answer correctness"
    }
)

print("✓ Pairwise comparison completed!")
print(f"\n📊 View comparison results in LangSmith UI")
print(f"   Comparison type: {type(comparison_results).__name__}")

# RAGAS with langSmith

!pip install ragas

In [ ]:
# RAGAS Metrics: Ground Truth Requirements
"""
Ground Truth (GT) = Reference/correct answer for a question

How to collect GT:
  • Synthetic: LLM generates Q&A from docs (RAGAS TestsetGenerator)
  • Manual: Subject matter experts create reference answers
  • Existing: FAQs, support tickets, documentation Q&As
  • Sampling: Validated production queries collected over time

Testing phase validation:
  • SMEs review synthetic GT for accuracy
  • Cross-validation with multiple annotators
  • Benchmark against known good answers
  • Iterative refinement based on RAG performance

┌──────────────────────┬────────────────┬──────────────────────────────┐
│ Metric               │ Needs GT?      │ Measures                     │
├──────────────────────┼────────────────┼──────────────────────────────┤
│ Faithfulness         │ ❌ NO          │ Answer grounded in context   │
│ Answer Relevancy     │ ❌ NO          │ Answer relevance to question │
│ Context Precision    │ ✅ YES         │ Relevant docs ranked higher  │
│ Context Recall       │ ✅ YES         │ GT info in retrieved context │
└──────────────────────┴────────────────┴──────────────────────────────┘

                    Do you have Ground Truth?
                                |
                +---------------+---------------+
                |                               |
               NO                              YES
                |                               |
          PRODUCTION                        TESTING
                |                               |
         Use 2 Metrics:                  Use ALL 4 Metrics:
         ✅ Faithfulness                 ✅ Faithfulness
         ✅ Answer Relevancy             ✅ Answer Relevancy
                                         ✅ Context Precision
                                         ✅ Context Recall
"""

In [ ]:
# ============================================================================
# RAGAS Evaluation Without Dataset: Fetch, Chunk, Embed & Evaluate
# ============================================================================

# Step 1: Fetch Wikipedia document
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
import sys
sys.path.append('/Users/vinotganesan/Learning/LLM & AGENTS/RAG')
from ml_server_embedding import get_embeddings

loader = WikipediaLoader(query="Machine Learning", load_max_docs=1)
raw_docs = loader.load()
print(f"✅ Fetched: {raw_docs[0].metadata['title']} ({len(raw_docs[0].page_content)} chars)")

# Step 2: Chunk documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
doc_chunks = text_splitter.split_documents(raw_docs)
print(f"✅ Created {len(doc_chunks)} chunks")

# Step 3: Create vector store
ml_vectorstore = Chroma.from_documents(
    documents=doc_chunks,
    embedding=get_embeddings(),
    collection_name="ml_wikipedia_ragas"
)
ml_retriever = ml_vectorstore.as_retriever(search_kwargs={"k": 4})
print(f"✅ Vector store ready with {len(doc_chunks)} embeddings")

# Step 4: Build RAG system
import openai
from langsmith import traceable
from langsmith.wrappers import wrap_openai

class SimpleRAG:
    def __init__(self, retriever):
        self.retriever = retriever
        self.client = wrap_openai(openai.Client(base_url=base_url, api_key=api_key))

    @traceable()
    def get_answer(self, question: str) -> dict:
        docs = self.retriever.invoke(question)
        contexts = [doc.page_content for doc in docs]

        response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": f"Answer concisely using this context:\n{' '.join(contexts)}"},
                {"role": "user", "content": question}
            ],
            temperature=0
        )

        return {"answer": response.choices[0].message.content, "contexts": contexts}

rag_system = SimpleRAG(ml_retriever)
print("✅ RAG system initialized")

# Step 5: Generate synthetic test questions
from ragas.testset import TestsetGenerator
from ragas.llms import llm_factory
import pandas as pd

openai_client = openai.Client(base_url=base_url, api_key=api_key)
generator_llm = llm_factory(model="gpt-4o", client=openai_client)

test_generator = TestsetGenerator(llm=generator_llm, embedding_model=get_embeddings())

print("🔄 Generating test questions...")
try:
    testset = test_generator.generate_with_langchain_docs(doc_chunks[:20], testset_size=5)
    testset_df = testset.to_pandas()
    print(f"✅ Generated {len(testset_df)} test cases")
except Exception as e:
    print(f"⚠️  Fallback to manual questions ({str(e)[:50]}...)")
    testset_df = pd.DataFrame({
        "user_input": ["What is machine learning?", "What are the main types of machine learning?"],
        "reference": [
            "Machine learning is a field of study that gives computers the ability to learn without being explicitly programmed.",
            "The main types are supervised learning, unsupervised learning, and reinforcement learning."
        ]
    })

# Step 6: Create LangSmith dataset - Use existing client from earlier cell
dataset_name = "RAG_ML_Wikipedia_RAGAS"

try:
    dataset = client.read_dataset(dataset_name=dataset_name)
    print(f"✅ Using existing dataset: {dataset_name}")
except:
    dataset = client.create_dataset(dataset_name=dataset_name, description="ML Wikipedia RAG evaluation with RAGAS")

    # Add examples to dataset
    inputs_list = [{"question": q} for q in testset_df['user_input'].tolist()]
    outputs_list = [{"answer": a} for a in testset_df.get('reference', [''] * len(testset_df)).tolist()]

    client.create_examples(inputs=inputs_list, outputs=outputs_list, dataset_id=dataset.id)
    print(f"✅ Created dataset with {len(inputs_list)} examples")

# Step 7: RAGAS evaluation function with LangSmith
from ragas import evaluate as ragas_evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
from datasets import Dataset
from langsmith.schemas import Run, Example

def predict_rag_answer(example: dict) -> dict:
    """RAG prediction for LangSmith evaluation"""
    question = example["question"]
    response = rag_system.get_answer(question)
    return {"answer": response["answer"], "contexts": response["contexts"]}

# RAGAS evaluators for LangSmith
ragas_llm = llm_factory(model="gpt-4o-mini", client=openai_client)

def ragas_faithfulness_evaluator(run: Run, example: Example) -> dict:
    """Faithfulness evaluator (no GT needed)"""
    try:
        eval_data = Dataset.from_dict({
            "question": [example.inputs.get("question", "")],
            "answer": [run.outputs.get("answer", "")],
            "contexts": [run.outputs.get("contexts", [])]
        })
        result = ragas_evaluate(eval_data, metrics=[Faithfulness(llm=ragas_llm)])
        score = result.to_pandas()['faithfulness'].mean()
        return {"key": "ragas_faithfulness", "score": score}
    except Exception as e:
        return {"key": "ragas_faithfulness", "score": None, "comment": str(e)}

def ragas_answer_relevancy_evaluator(run: Run, example: Example) -> dict:
    """Answer relevancy evaluator (no GT needed)"""
    try:
        eval_data = Dataset.from_dict({
            "question": [example.inputs.get("question", "")],
            "answer": [run.outputs.get("answer", "")],
            "contexts": [run.outputs.get("contexts", [])]
        })
        result = ragas_evaluate(eval_data, metrics=[AnswerRelevancy(llm=ragas_llm, embeddings=get_embeddings())])
        score = result.to_pandas()['answer_relevancy'].mean()
        return {"key": "ragas_answer_relevancy", "score": score}
    except Exception as e:
        return {"key": "ragas_answer_relevancy", "score": None, "comment": str(e)}

def ragas_context_precision_evaluator(run: Run, example: Example) -> dict:
    """Context precision evaluator (GT required)"""
    ground_truth = example.outputs.get("answer", "")
    if not ground_truth:
        return {"key": "ragas_context_precision", "score": None, "comment": "No ground truth"}

    try:
        eval_data = Dataset.from_dict({
            "question": [example.inputs.get("question", "")],
            "answer": [run.outputs.get("answer", "")],
            "contexts": [run.outputs.get("contexts", [])],
            "ground_truth": [ground_truth]
        })
        result = ragas_evaluate(eval_data, metrics=[ContextPrecision(llm=ragas_llm)])
        score = result.to_pandas()['context_precision'].mean()
        return {"key": "ragas_context_precision", "score": score}
    except Exception as e:
        return {"key": "ragas_context_precision", "score": None, "comment": str(e)}

def ragas_context_recall_evaluator(run: Run, example: Example) -> dict:
    """Context recall evaluator (GT required)"""
    ground_truth = example.outputs.get("answer", "")
    if not ground_truth:
        return {"key": "ragas_context_recall", "score": None, "comment": "No ground truth"}

    try:
        eval_data = Dataset.from_dict({
            "question": [example.inputs.get("question", "")],
            "answer": [run.outputs.get("answer", "")],
            "contexts": [run.outputs.get("contexts", [])],
            "ground_truth": [ground_truth]
        })
        result = ragas_evaluate(eval_data, metrics=[ContextRecall(llm=ragas_llm)])
        score = result.to_pandas()['context_recall'].mean()
        return {"key": "ragas_context_recall", "score": score}
    except Exception as e:
        return {"key": "ragas_context_recall", "score": None, "comment": str(e)}

# Step 8: Run LangSmith evaluation with RAGAS
from langsmith.evaluation import evaluate

print("\n🔄 Running LangSmith evaluation with RAGAS metrics...")

evaluation_result = evaluate(
    predict_rag_answer,
    data=dataset_name,
    evaluators=[
        ragas_faithfulness_evaluator,
        ragas_answer_relevancy_evaluator,
        ragas_context_precision_evaluator,
        ragas_context_recall_evaluator,
    ],
    experiment_prefix="ragas-ml-wikipedia",
    metadata={
        "evaluation_type": "RAGAS with LangSmith",
        "document_source": "Wikipedia ML article",
        "embeddings": "custom get_embeddings()",
        "llm": "gpt-4o-mini"
    }
)

print(f"\n✅ Evaluation complete!")
print(f"📊 Experiment: {evaluation_result.experiment_name}")
print(f"🔗 View results: {evaluation_result.experiment_url}")

# Display summary
results_df = evaluation_result.to_pandas()
print("\n📊 RAGAS Metrics Summary:")
for metric in ['ragas_faithfulness', 'ragas_answer_relevancy', 'ragas_context_precision', 'ragas_context_recall']:
    if metric in results_df.columns:
        mean_score = results_df[metric].mean()
        print(f"   {metric.replace('ragas_', '').replace('_', ' ').title()}: {mean_score:.4f}")

print("\n✅ Results published to LangSmith!")

!pip install wikipedia

In [ ]:
# Debug: Check RAGAS TestsetGenerator method availability
from ragas.testset import TestsetGenerator
import inspect

print("📊 Available TestsetGenerator methods:")
methods = [m for m in dir(TestsetGenerator) if not m.startswith('_')]
for method in methods:
    print(f"   - {method}")

print("\n🔍 TestsetGenerator signature:")
print(inspect.signature(TestsetGenerator.__init__))

# Check generate methods
generate_methods = [m for m in methods if 'generate' in m.lower()]
print(f"\n🎯 Generate methods available: {generate_methods}")

# Let's try to see what parameters the generate method expects
if 'generate' in methods:
    print("\n📝 generate() method signature:")
    try:
        print(inspect.signature(TestsetGenerator.generate))
    except:
        print("   Could not retrieve signature")

if 'generate_with_langchain_docs' in methods:
    print("\n📝 generate_with_langchain_docs() method signature:")
    try:
        print(inspect.signature(TestsetGenerator.generate_with_langchain_docs))
    except:
        print("   Could not retrieve signature")